# DART 기반 매출예측 → FCFF Valuation — `dart_fcff_valuation_v1`

**파이프라인**

```
korea_fs_data_from_DART_V2 (dart_korea_fs_loader_v6 적재본)
     │  PART 1  DART 계정 → 표준키 매핑 · 분기화(누적/분기 자동 판별)
     ▼
분기 매출 시계열 ──► PART 2  SARIMA · ETS · Theta · Ensemble (Korea_revenue_forecast_v4 준용)
     │                       └─► korea_revenue_forecast_from_DART  (forecast_date 별 vintage 저장)
     ▼
PART 3  DartDCFModel  (korea_fcff_individual_valuation_v1 = v8 모형 + US v13 가드 그대로)
     │            부족 항목은 DART 데이터로 추정 (이자비용→금융비용, 주식수→시총/주가 등)
     ▼
PART 4  Excel 출력  →  …\FCFF_RIM_재무데이터\{ticker}_{기업명}_FCFF_DART_{YYYYMMDD}.xlsx   (DB 저장 없음)
```

| 준용 원본 | 이 노트북에서의 역할 |
|---|---|
| `dart_korea_fs_loader_v6` | 테이블 스키마(`korea_fs_data_from_DART_V2`)·quarter 라벨(Q1/H1/Q3/FY)·단위(원) |
| `Korea_revenue_forecast_v4` | `_prepare_series` / `_run_models` / long 변환 / 배치 저장 — 로직 동일, 원천·저장 테이블만 변경 |
| `korea_fcff_individual_valuation_v1` | `KoreaDCFModel v8` 전체 로직 — 데이터 로더만 DART 로 교체 |

**DART 원천의 특성 (DG 와 다른 점)**
- 금액 단위 **원** (DG 는 천원) → `FS_UNIT_MULTIPLIER = 1`
- 손익계산서(IS/CIS) `thstrm_amount` 는 보고서별 3개월치, 현금흐름표(CF) 는 누적치가 일반적 → `_detect_flow_mode()` 로 종목별 자동 판별 후 순수 분기값 산출
- 주식수 항목이 재무제표 API 에 없음 → `시가총액(ks_listed_company_daily_marketcap) ÷ 현재가` 로 추정
- 계정 ID 가 표준(ifrs-full_*)·비표준("-표준계정코드 미사용-") 혼재 → `DART_ACCOUNT_MAP` 이 **account_id 우선, account_nm 정규식 보조** 로 매핑

실행 순서: **Cell 1 → Cell 2(입력) → 위에서 아래로 순서대로**

## PART 1 · 환경
### Cell 1 · 경로 자동 감지 (노트북 / 데스크탑 공용)

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    """DATA 폴더의 부모를 프로젝트 루트로 sys.path 에 등록 (루트 + DATA 둘 다)."""
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            for q in (root, str(p / "DATA")):
                if q not in sys.path:
                    sys.path.insert(0, q)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(os.path.join(cand, "DATA")):
            for q in (cand, os.path.join(cand, "DATA")):
                if q not in sys.path:
                    sys.path.insert(0, q)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 수정하세요.")

_ROOT = _setup_path()
print(f"[확인] DATA 경로 : {os.path.join(_ROOT, 'DATA')}")

### Cell 2 · ★ 입력 변수 (여기만 수정)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  ★★★ Control Panel — 이 노트북의 모든 사용자 입력은 여기 한 곳 ★★★
# ═══════════════════════════════════════════════════════════════
from datetime import date

# ── ① 평가 대상 ────────────────────────────────────────────────
TICKERS = [                 # DART 형식 6자리 ('A' 접두사 있어도 자동 제거)
    "000660",               #   SK하이닉스
    # "278470",             #   에이피알
]
VERBOSE = True

# ── ② 출력 폴더 (요구사항 7) — 첫 번째로 존재하는 부모 폴더 채택 ──
_EXPORT_DIR_CANDIDATES = [
    r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터",          # 랩탑
    r"C:\Users\Hoyoung_Park\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터",   # 데스크탑
]
EXPORT_FILE_TAG = "FCFF_DART"       # 파일명에 DART 기반임을 표시 (요구사항 7)
CORP_NAMES = {                      # 파일명용 기업명 (없으면 시총 테이블에서 시도, 실패 시 'NA')
    "000660": "SK하이닉스", "005930": "삼성전자", "278470": "에이피알",
}

# ── ③ DB 테이블 ────────────────────────────────────────────────
TABLE_DART_FS      = "korea_fs_data_from_DART_V2"        # 원천 (loader_v6)
TABLE_DART_FC      = "korea_revenue_forecast_from_DART"  # ★ 신규: DART 매출예측 (vintage)
TABLE_MARKETCAP    = "ks_listed_company_daily_marketcap"
TABLE_PRICE        = "KSE_Price"

# ── ④ 매출 예측 (Korea_revenue_forecast_v4 준용) ─────────────
FORECAST_QUARTERS  = 8          # 예측 분기 수
MIN_DATA_PERIODS   = 20         # 최소 연속 분기 (DART 는 2015~ 이므로 24 → 20 완화)
ENSEMBLE_AGG       = "median"   # 'median' | 'mean'
FORECAST_DATE      = date.today()   # ★ 계측일 (vintage key). 과거 날짜 지정 시 그 날짜로 저장
RERUN_FORECAST     = True       # False → 같은 forecast_date 에 이미 저장된 예측이 있으면 재예측 생략
FORECAST_MODULE_CANDIDATES = ["universal_ts_forecast_function_v3",
                              "universal_ts_forecast_function_v2",
                              "universal_ts_forecast_function"]

# ── ⑤ DART 분기화 방식 ────────────────────────────────────────
#   'auto'       : 연도별 (Q1+H1+Q3)/FY 비율로 누적/분기 자동 판별 (권장)
#   'quarter'    : thstrm_amount 가 각 보고서의 3개월치 (Q4 = FY − Q1−Q2−Q3)
#   'cumulative' : thstrm_amount 가 누적치 (Q2 = H1 − Q1 …)
IS_FLOW_MODE = "auto"
CF_FLOW_MODE = "auto"

# ── ⑥ 모델 파라미터 (korea_fcff_individual_valuation_v1 과 동일) ──
FORECAST_HORIZON     = FORECAST_QUARTERS
MIN_HISTORY          = 16
WINSORIZE_LIMITS     = (0.05, 0.95)
GDP_GROWTH           = 0.04
G_FLOOR_MIN          = -0.20
G_CEIL               = 1.00
OLS_MIN_R2           = 0.30
OLS_MIN_SAMPLES      = 16
MIN_REVENUE_QUARTERS = MIN_DATA_PERIODS

# ── ⑦ WACC / ERP ───────────────────────────────────────────────
ERP_METHOD       = "damodaran_floor"
DAMODARAN_ERP_KR = 0.07
GEO_FLOOR        = 0.07
RF_FALLBACK      = 0.035
RD_DEFAULT       = 0.045
RD_ANNUALIZE_QUARTERLY = True   # 분기 이자비용 ×4 후 평균부채로 나눔 (v1 은 미연율화 → Rf floor 에 걸리던 부분 보정)
WACC_FLOOR       = 0.05
WACC_CAP         = 0.20

# ── ⑧ v8 Validity 가드 ────────────────────────────────────────
V8_RD_FLOOR_RF      = True
V8_WACC_FLOOR_ABS   = 0.05
V8_MIN_TV_SPREAD    = 0.02
V8_TERMINAL_RF_CAP  = True
V8_TV_FCFF_MULT_CAP = 35.0
V8_SANITY_GUARD     = True
V8_SANITY_MC_RATIO  = 30.0
V8_RHO_ADAPTIVE_CAP = True

# ── ⑨ Net Debt / NWC 구성 (표준키) ───────────────────────────
DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds", "long_term_debt", "lease_liab"]
CASH_KEYS = ["cash", "short_term_invest"]
USE_OPERATING_NWC = True

# ── ⑩ 단위 ────────────────────────────────────────────────────
MARKETCAP_UNIT_MULTIPLIER = 1_000_000   # 시총 테이블: 백만원 → 원
FS_UNIT_MULTIPLIER        = 1           # ★ DART 는 원 단위 (DG 천원과 다름)

# ── 출력 폴더 확정 ────────────────────────────────────────────
def _pick_export_dir() -> Path:
    for c in _EXPORT_DIR_CANDIDATES:
        p = Path(c)
        if p.exists() or p.parent.exists():
            p.mkdir(parents=True, exist_ok=True)
            return p
    p = Path.home() / "FCFF_RIM_재무데이터"
    p.mkdir(parents=True, exist_ok=True)
    print(f"[WARN] 후보 경로 없음 → {p} 사용")
    return p

EXPORT_DIR = _pick_export_dir()
TICKERS = [str(t).strip().upper().lstrip("A").zfill(6) for t in TICKERS]

print("[OK] Control Panel")
print(f"  대상 종목      : {TICKERS}")
print(f"  Excel 폴더     : {EXPORT_DIR}")
print(f"  forecast_date  : {FORECAST_DATE}   예측 {FORECAST_QUARTERS}Q   최소 {MIN_DATA_PERIODS}Q   앙상블 {ENSEMBLE_AGG}")
print(f"  원천 → 예측 DB : {TABLE_DART_FS} → {TABLE_DART_FC}")

### Cell 3 · Import & DB 연결

In [ ]:
import gc, re, math, time, traceback, importlib
from datetime import datetime, timedelta
from typing import Optional, Dict, Any, List, Tuple, Union

import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
from tqdm.auto import tqdm
from sqlalchemy import text as _sa_text

# ── 내부 모듈 ───────────────────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    to_dg_ticker, to_price_ticker, get_pymysql_conn,
    load_korea_marketcap_latest, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, DataQualityReport,
)

# 예측 모듈: 후보 순서대로 import (v4 노트북과 동일한 폴백 방식)
_fc_mod = None
for _name in FORECAST_MODULE_CANDIDATES:
    try:
        _fc_mod = importlib.import_module(_name)
        FORECAST_MODULE = _name
        break
    except ModuleNotFoundError:
        continue
if _fc_mod is None:
    raise ImportError(f"예측 모듈을 찾을 수 없습니다: {FORECAST_MODULE_CANDIDATES}")
forecast_sarima = _fc_mod.forecast_sarima
forecast_ets    = _fc_mod.forecast_ets
forecast_theta  = _fc_mod.forecast_theta
infer_freq_alias          = getattr(_fc_mod, "infer_freq_alias", None)
seasonal_periods_from_freq = getattr(_fc_mod, "seasonal_periods_from_freq", None)
clear_memory              = getattr(_fc_mod, "clear_memory", lambda: gc.collect())

def log(tag: str, msg: str):
    print(f"[{datetime.now():%H:%M:%S}][{tag}] {msg}", flush=True)

db_info = get_db_info()
engine  = get_engine(db_info)
with engine.connect() as c:
    c.execute(_sa_text("SELECT 1"))
log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
log("MOD", f"예측 모듈 = {FORECAST_MODULE}")

### Cell 4 · 시장 파라미터 (Rf · KOSPI · E(Rm)) — v1 동일

In [ ]:
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
KOSPI_PX = load_kospi_series(start_date=(datetime.today() - timedelta(days=365*12)).strftime("%Y-%m-%d"))
mkt = estimate_market_return(method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX, years=10,
                             damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR)
E_RM, ERP = mkt["e_rm"], mkt["erp"]
log("MKT", f"Rf={RF:.3%} ({RF_SOURCE})  ERP={ERP:.3%}  E(Rm)={E_RM:.3%}  KOSPI={KOSPI_PX.iloc[-1]:,.1f}")

### Cell 5 · DART 계정 → 표준키 매핑 & 분기화 로더

`DART_ACCOUNT_MAP[std_key]` = 컴포넌트 리스트. 각 컴포넌트는 **account_id 후보(우선) → account_nm 정규식(보조)** 순으로
첫 매칭 계정 하나만 채택(중복합산 방지)하고, 컴포넌트끼리는 합산(예: 리스부채 유동+비유동).

`flow` : `stock`(BS 시점값) / `is`(손익 흐름) / `cf`(현금흐름 흐름). 흐름 항목은 `_detect_flow_mode()` 결과에 따라 순수 분기값으로 변환.

In [ ]:
# ── 표준키 정의 ────────────────────────────────────────────────
#   comp : 컴포넌트 리스트(합산). 각 컴포넌트의 ids 후보는 기간별로 coalesce (택소노미 변경 대응:
#          ifrs-full_X 를 쓰면 ifrs_X 도 자동 후보). nm 은 ids 모두 없을 때만 정규식 보조.
#   alts : 대안 그룹 리스트 — 앞 그룹이 하나도 안 잡힐 때만 다음 그룹 사용 (중복합산 방지)
DART_ACCOUNT_MAP: Dict[str, Dict[str, Any]] = {
    # ── 손익 (IS → CIS) ──────────────────────────────────────
    "revenue":          {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["ifrs-full_Revenue"], "nm": r"^(매출액|매출|수익\(매출액\)|영업수익)$"}]},
    "cost_of_sales":    {"flow": "is", "sj": ["IS", "CIS"], "comp": [{"ids": ["ifrs-full_CostOfSales"], "nm": r"^매출원가$"}]},
    "operating_income": {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["dart_OperatingIncomeLoss", "ifrs-full_ProfitLossFromOperatingActivities"], "nm": r"^영업이익(\(손실\))?$"}]},
    "pretax_income":    {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["ifrs-full_ProfitLossBeforeTax"], "nm": r"^법인세(비용)?차감전(순)?(이익|손익)"}]},
    "tax_expense":      {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["ifrs-full_IncomeTaxExpenseContinuingOperations"], "nm": r"^법인세(비용|비용\(수익\)|비용\(이익\)|수익\(비용\))$"}]},
    "net_income":       {"flow": "is", "sj": ["IS", "CIS"], "alts": [
        [{"ids": ["ifrs-full_ProfitLoss"], "nm": r"^(연결)?(당기|분기|반기|당분기|당반기)?순(이익|손익|손실)(\(손실\)|\(이익\))?$"}],
        [{"ids": ["ifrs-full_ProfitLossAttributableToOwnersOfParent"]},          # 지배 + 비지배 합산
         {"ids": ["ifrs-full_ProfitLossAttributableToNoncontrollingInterests"]}],
    ]},
    "net_income_parent": {"flow": "is", "sj": ["IS", "CIS"], "comp": [{"ids": ["ifrs-full_ProfitLossAttributableToOwnersOfParent"]}]},
    "interest_expense": {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["dart_InterestExpenseFinanceCosts", "ifrs-full_InterestExpense"], "nm": r"^이자비용$"}]},
    "finance_costs":    {"flow": "is", "sj": ["IS", "CIS"], "comp": [{"ids": ["ifrs-full_FinanceCosts"], "nm": r"^(금융비용|금융원가)$"}]},
    # ── 현금흐름 ─────────────────────────────────────────────
    "da_cf":            {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_AdjustmentsForDepreciationExpense", "dart_DepreciationExpenseCashFlow"], "nm": r"^감가상각비$"}]},
    "intangible_amort_cf": {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_AdjustmentsForAmortisationExpense", "dart_AmortisationExpenseCashFlow"], "nm": r"^무형자산상각비?$"}]},
    "capex_tangible":   {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities", "dart_PurchaseOfPropertyPlantAndEquipment"],
         "nm": r"^유형자산의?(취득|증가)$"}]},
    "capex_intangible": {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities", "dart_PurchaseOfIntangibleAssets"],
         "nm": r"^무형자산의?(취득|증가)$"}]},
    "interest_paid_cf": {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_InterestPaidClassifiedAsOperatingActivities", "ifrs-full_InterestPaidClassifiedAsFinancingActivities"],
         "nm": r"^이자(의)?지급$"}]},
    "cfo":              {"flow": "cf", "sj": ["CF"], "comp": [{"ids": ["ifrs-full_CashFlowsFromUsedInOperatingActivities"], "nm": r"^영업활동현금흐름$"}]},
    "dividends_paid":   {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_DividendsPaidClassifiedAsFinancingActivities", "dart_DividendsPaid"], "nm": r"^(현금)?배당금(의)?지급$"}]},
    # ── 재무상태표 ───────────────────────────────────────────
    "cash":             {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_CashAndCashEquivalents"], "nm": r"^현금및현금성자산$"}]},
    "short_term_invest": {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermDepositsNotClassifiedAsCashEquivalents", "ifrs-full_ShorttermDepositsNotClassifiedAsCashEquivalents"], "nm": r"^단기금융상품$"},
        {"ids": ["ifrs-full_CurrentInvestments", "dart_CurrentAvailableForSaleFinancialAssets", "dart_ShortTermTradingFinancialAssets"], "nm": r"^단기투자자산$"}]},
    "receivables":      {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermTradeReceivable", "ifrs-full_CurrentTradeReceivables", "ifrs-full_TradeAndOtherCurrentReceivables"],
         "nm": r"^(매출채권|매출채권및기타채권|매출채권및기타유동채권)$"}]},
    "inventories":      {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_Inventories"], "nm": r"^재고자산$"}]},
    "prepaid_expenses": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["dart_ShortTermPrepaidExpenses"], "nm": r"^선급비용$"}]},
    "payables":         {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermTradePayables", "ifrs-full_TradeAndOtherCurrentPayablesToTradeSuppliers", "ifrs-full_TradeAndOtherCurrentPayables"],
         "nm": r"^(매입채무|매입채무및기타채무|매입채무및기타유동채무)$"}]},
    "accrued_expenses": {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermAccruedExpenses", "dart_CurrentNontradePayables"], "nm": r"^(미지급비용|기타지급채무)$"}]},
    "other_payables":   {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["dart_ShortTermOtherPayables"], "nm": r"^(미지급금|단기미지급금)$"}]},
    "advances_received": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["dart_ShortTermAdvancesCustomers"], "nm": r"^선수금$"}]},
    "contract_liabilities": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_CurrentContractLiabilities"], "nm": r"^(계약부채|유동계약부채)$"}]},
    "short_term_debt":  {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_ShorttermBorrowings", "dart_ShortTermBorrowings", "ifrs-full_CurrentBorrowingsAndCurrentPortionOfNoncurrentBorrowings"],
         "nm": r"^(단기차입금|단기차입금및유동성장기차입금)$"}]},
    "current_lt_debt":  {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_CurrentPortionOfLongtermBorrowings", "dart_CurrentPortionOfLongTermBorrowingsAndDebentures"],
         "nm": r"^유동성(장기차입금|장기부채|사채|장기차입부채)"}]},
    "bonds":            {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_BondsIssued", "dart_BondsIssued", "ifrs-full_DebenturesIssued"], "nm": r"^(사채|비유동사채|회사채)$"}]},
    "long_term_debt":   {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_LongtermBorrowings", "dart_LongTermBorrowingsGross"], "nm": r"^(장기차입금|비유동차입금)$"}]},
    "lease_liab":       {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_CurrentLeaseLiabilities", "dart_CurrentLeaseLiabilities"], "nm": r"^(유동리스부채|리스부채\(유동\))$"},
        {"ids": ["ifrs-full_NoncurrentLeaseLiabilities", "dart_NoncurrentLeaseLiabilities"], "nm": r"^(비유동리스부채|리스부채\(비유동\)|장기리스부채)$"}]},
    "total_equity":     {"flow": "stock", "sj": ["BS"], "alts": [
        [{"ids": ["ifrs-full_Equity"], "nm": r"^자본총계$"}],
        [{"ids": ["ifrs-full_EquityAttributableToOwnersOfParent"]}, {"ids": ["ifrs-full_NoncontrollingInterests"]}],
    ]},
    "equity_parent":    {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_EquityAttributableToOwnersOfParent"]}]},
    "total_assets":     {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_Assets"], "nm": r"^자산총계$"}]},
    "total_liabilities": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_Liabilities"], "nm": r"^부채총계$"}]},
    "ppe":              {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_PropertyPlantAndEquipment"], "nm": r"^유형자산$"}]},
    "intangibles":      {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_IntangibleAssetsOtherThanGoodwill", "ifrs-full_IntangibleAssetsAndGoodwill", "dart_GoodwillGross"], "nm": r"^무형자산$"}]},
}

_Q_ORDER = {"Q1": 1, "H1": 2, "Q3": 3, "FY": 4}


def _expand_ids(ids: List[str]) -> List[str]:
    """ifrs-full_X → [ifrs-full_X, ifrs_X] 자동 확장 (구 택소노미 대응)."""
    out = []
    for i in ids:
        out.append(i)
        if i.startswith("ifrs-full_"):
            out.append("ifrs_" + i[len("ifrs-full_"):])
    return list(dict.fromkeys(out))


def load_dart_long(ticker: str, db_info: dict, table: str = TABLE_DART_FS) -> pd.DataFrame:
    """DART 원천 long 테이블 조회 (ticker 6자리). SQLAlchemy engine 경유 (pymysql DictCursor 와 read_sql 충돌 회피)."""
    sql = _sa_text(f"""
        SELECT corp_code, bsns_year, reprt_code, quarter, sj_div, account_id, account_nm,
               thstrm_amount, report_date, ticker
        FROM {table}
        WHERE ticker = :t AND thstrm_amount IS NOT NULL
        ORDER BY bsns_year, reprt_code
    """)
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params={"t": ticker})
    if df.empty:
        return df
    # 방어: 헤더가 데이터로 들어온 경우 제거
    df = df[df["report_date"].astype(str) != "report_date"].copy()
    df["account_nm_norm"] = df["account_nm"].astype(str).str.replace(r"\s+", "", regex=True)
    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
    df["bsns_year"] = pd.to_numeric(df["bsns_year"], errors="coerce").astype(int)
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    return df


def _resolve_component(df: pd.DataFrame, comp: Dict, sj_list: List[str]) -> List[Tuple[str, str]]:
    """컴포넌트 → [(token, how), ...]  (기간별 coalesce 순서). ids 는 우선순위대로 모두 채택, 없으면 nm 정규식."""
    sub = df[df["sj_div"].isin(sj_list)]
    toks = [(aid, "id") for aid in _expand_ids(comp.get("ids", [])) if (sub["account_id"] == aid).any()]
    if not toks and comp.get("nm"):
        m = sub[sub["account_nm_norm"].str.match(comp["nm"])]
        if not m.empty:
            grp = m.groupby(["account_id", "account_nm"]).size().sort_values(ascending=False)
            toks = [(f"NM::{aid}::{anm}", "nm") for (aid, anm) in grp.index]
    return toks


def _component_series(df: pd.DataFrame, toks: List[Tuple[str, str]], sj_list: List[str]) -> pd.DataFrame:
    """토큰 리스트를 우선순위대로 coalesce → [bsns_year, quarter, report_date, value]"""
    sub = df[df["sj_div"].isin(sj_list)]
    sub = sub.assign(_sj_rank=sub["sj_div"].map({s: i for i, s in enumerate(sj_list)}))
    parts = []
    for rank, (tok, _) in enumerate(toks):
        if tok.startswith("NM::"):
            _, aid, anm = tok.split("::", 2)
            x = sub[(sub["account_id"] == aid) & (sub["account_nm"] == anm)]
        else:
            x = sub[sub["account_id"] == tok]
        parts.append(x.assign(_rank=rank))
    if not parts:
        return pd.DataFrame(columns=["bsns_year", "quarter", "report_date", "value"])
    allx = pd.concat(parts).sort_values(["bsns_year", "quarter", "_rank", "_sj_rank"])
    allx = allx.drop_duplicates(["bsns_year", "quarter"])       # 기간별 첫 우선순위 채택
    return allx[["bsns_year", "quarter", "report_date", "thstrm_amount"]].rename(columns={"thstrm_amount": "value"})


def _detect_flow_mode(pivot: pd.DataFrame, verbose: bool = False, label: str = "") -> str:
    """
    연도별 (Q1+H1+Q3)/FY 비율의 중앙값으로 누적/분기 판별.
      분기값이면 ≈ 0.75, 누적값이면 ≈ 1.5 (1/4+2/4+3/4).  임계 1.10.
    pivot: index=bsns_year, columns=Q1/H1/Q3/FY
    """
    need = ["Q1", "H1", "Q3", "FY"]
    if not all(c in pivot.columns for c in need):
        return "quarter"
    full = pivot.dropna(subset=need)
    full = full[(full[need].abs() > 0).all(axis=1)]
    if full.empty:
        return "quarter"
    ratio = ((full["Q1"] + full["H1"] + full["Q3"]) / full["FY"]).abs()
    med = float(ratio.median())
    mode = "cumulative" if med > 1.10 else "quarter"
    if verbose:
        log("FLOW", f"{label}: median (Q1+H1+Q3)/FY = {med:.2f} (n={len(full)}) → {mode}")
    return mode


def _flows_to_quarterly(pivot: pd.DataFrame, mode: str) -> pd.DataFrame:
    """index=bsns_year, columns Q1/H1/Q3/FY → 순수 분기값 columns Q1..Q4"""
    out = pd.DataFrame(index=pivot.index, columns=["Q1", "Q2", "Q3", "Q4"], dtype=float)
    q1 = pivot.get("Q1"); h1 = pivot.get("H1"); q3 = pivot.get("Q3"); fy = pivot.get("FY")
    if mode == "cumulative":
        out["Q1"] = q1
        out["Q2"] = h1 - q1 if (h1 is not None and q1 is not None) else np.nan
        out["Q3"] = q3 - h1 if (q3 is not None and h1 is not None) else np.nan
        out["Q4"] = fy - q3 if (fy is not None and q3 is not None) else np.nan
    else:  # quarter
        out["Q1"] = q1
        out["Q2"] = h1
        out["Q3"] = q3
        out["Q4"] = fy - (q1 + h1 + q3) if all(x is not None for x in (fy, q1, h1, q3)) else np.nan
    return out


def _quarter_end(year: int, q: int) -> pd.Timestamp:
    return pd.Period(f"{year}Q{q}", freq="Q").to_timestamp(how="end").normalize()


def load_dart_financials_wide(ticker: str, db_info: dict, verbose: bool = False,
                              is_mode: str = IS_FLOW_MODE, cf_mode: str = CF_FLOW_MODE
                              ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    DART long → 분기 wide (index=분기말 Timestamp, columns=표준키, 단위 원).
    Returns (wide, mapping_df)  mapping_df: 표준키별 채택 계정·매칭방식·flow 모드.
    """
    ticker = str(ticker).upper().lstrip("A").zfill(6)
    df = load_dart_long(ticker, db_info)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    def _comp_groups(spec: Dict) -> List[List[Dict]]:
        return spec["alts"] if "alts" in spec else [spec["comp"]]

    def _resolve_group(spec: Dict) -> List[Tuple[Dict, List[Tuple[str, str]]]]:
        """alts 중 하나라도 잡히는 첫 그룹의 [(comp, toks)] 반환"""
        for grp in _comp_groups(spec):
            got = [(c, _resolve_component(df, c, spec["sj"])) for c in grp]
            if any(t for _, t in got):
                return got
        return []

    # ── 1) 흐름 모드 판별 (IS: revenue, CF: da_cf → capex_tangible → cfo) ──
    def _pivot_for(std_key: str) -> Optional[pd.DataFrame]:
        spec = DART_ACCOUNT_MAP[std_key]; got = _resolve_group(spec)
        for c, toks in got:
            if toks:
                s = _component_series(df, toks, spec["sj"])
                return s.pivot(index="bsns_year", columns="quarter", values="value")
        return None

    if is_mode == "auto":
        p = _pivot_for("revenue")
        is_mode = _detect_flow_mode(p, verbose, "IS(revenue)") if p is not None else "quarter"
    if cf_mode == "auto":
        cf_mode = "cumulative"
        for k in ("da_cf", "capex_tangible", "cfo"):
            p = _pivot_for(k)
            if p is not None:
                cf_mode = _detect_flow_mode(p.abs(), verbose, f"CF({k})"); break

    # ── 2) 표준키별 시계열 구성 ────────────────────────────────
    series_map: Dict[str, pd.Series] = {}
    mapping_rows = []
    for std_key, spec in DART_ACCOUNT_MAP.items():
        total: Optional[pd.Series] = None
        mode = "stock" if spec["flow"] == "stock" else (is_mode if spec["flow"] == "is" else cf_mode)
        for comp, toks in _resolve_group(spec):
            if not toks:
                continue
            s = _component_series(df, toks, spec["sj"])
            pv = s.pivot(index="bsns_year", columns="quarter", values="value")
            for lab in ("Q1", "H1", "Q3", "FY"):
                if lab not in pv.columns:
                    pv[lab] = np.nan
            if spec["flow"] == "stock":
                qdf = pd.DataFrame({"Q1": pv["Q1"], "Q2": pv["H1"], "Q3": pv["Q3"], "Q4": pv["FY"]}, index=pv.index)
            else:
                qdf = _flows_to_quarterly(pv, mode)
            long = qdf.reset_index().melt(id_vars=qdf.index.name or "index", var_name="qn", value_name="value")
            long.columns = ["bsns_year", "qn", "value"]
            long = long.dropna(subset=["value"])
            long["date"] = [_quarter_end(int(y), int(q[1])) for y, q in zip(long["bsns_year"], long["qn"])]
            ser = long.set_index("date")["value"].astype(float)
            total = ser if total is None else total.add(ser, fill_value=0.0)
            disp = " | ".join(t if how == "id" else t.split("::", 2)[2] + f"[{t.split('::', 2)[1]}]" for t, how in toks)
            mapping_rows.append({"std_key": std_key, "component": disp, "match": toks[0][1], "flow": spec["flow"], "mode": mode,
                                 "n_q": int(ser.notna().sum())})
        if total is not None:
            series_map[std_key] = total
        else:
            mapping_rows.append({"std_key": std_key, "component": "(없음)", "match": "-", "flow": spec["flow"], "mode": "-", "n_q": 0})

    wide = pd.DataFrame(series_map).sort_index().dropna(how="all")

    # ── 3) DART 기반 보완 추정 (요구사항 5) ────────────────────
    def _est(key, note, flow="derived"):
        mapping_rows.append({"std_key": key, "component": note, "match": "est", "flow": flow, "mode": "derived",
                             "n_q": int(wide[key].notna().sum()) if key in wide.columns else 0})
    def _missing(k):
        return k not in wide.columns or wide[k].dropna().empty
    def _fill(k, ser, note):
        if k in wide.columns:
            wide[k] = wide[k].where(wide[k].notna(), ser)
        else:
            wide[k] = ser
        _est(k, note)
    #   순이익 없음 → 세전이익 − 법인세
    if _missing("net_income") and not _missing("pretax_income") and "tax_expense" in wide.columns:
        _fill("net_income", wide["pretax_income"] - wide["tax_expense"].fillna(0), "← pretax_income − tax_expense (대체 추정)")
    #   자본총계 결측 기간 → 자산총계 − 부채총계
    if "total_assets" in wide.columns and "total_liabilities" in wide.columns:
        eq_alt = wide["total_assets"] - wide["total_liabilities"]
        if _missing("total_equity") or wide["total_equity"].isna().any():
            _fill("total_equity", eq_alt, "← total_assets − total_liabilities (결측 기간 보완)")
    #   이자비용 없음 → CF 이자의 지급 → 금융비용
    if _missing("interest_expense"):
        if not _missing("interest_paid_cf"):
            _fill("interest_expense", wide["interest_paid_cf"].abs(), "← CF 이자의 지급 (대체 추정)")
        elif not _missing("finance_costs"):
            _fill("interest_expense", wide["finance_costs"], "← finance_costs (대체 추정)")
    #   CapEx 없음 → ΔPPE + D&A
    if _missing("capex_tangible") and "ppe" in wide.columns:
        da = wide["da_cf"].fillna(0) if "da_cf" in wide.columns else 0.0
        _fill("capex_tangible", (wide["ppe"].diff() + da).clip(lower=0), "← ΔPPE + D&A (대체 추정)")
    #   D&A 없음 (DART 요약 CF 에는 조정항목 없음) → |CapEx| − ΔPPE  (처분 무시, ≥0)
    if _missing("da_cf") and "ppe" in wide.columns and not _missing("capex_tangible"):
        _fill("da_cf", (wide["capex_tangible"].abs() - wide["ppe"].diff()).clip(lower=0), "← |CapEx| − ΔPPE (대체 추정)")
    if _missing("intangible_amort_cf") and "intangibles" in wide.columns and not _missing("capex_intangible"):
        _fill("intangible_amort_cf", (wide["capex_intangible"].abs() - wide["intangibles"].diff()).clip(lower=0), "← |CapEx무형| − Δ무형자산 (대체 추정)")

    mapping_df = pd.DataFrame(mapping_rows)
    if verbose:
        n_ok = mapping_df[mapping_df["match"] != "-"]["std_key"].nunique()
        log(ticker, f"DART wide shape={wide.shape}  {wide.index.min().date()}~{wide.index.max().date()}  "
                    f"표준키 {n_ok}/{len(DART_ACCOUNT_MAP)} 매핑  IS={is_mode} CF={cf_mode}")
    return wide, mapping_df


def show_dart_account_coverage(ticker: str, db_info: dict = None):
    """개별 종목 점검: 표준키별 채택 계정 + 최근 8분기 값 (억원)."""
    db_info = db_info or get_db_info()
    wide, mp = load_dart_financials_wide(ticker, db_info, verbose=True)
    if wide.empty:
        print(f"[{ticker}] DART 데이터 없음"); return None, None
    print("\n[표준키 매핑]")
    print(mp.to_string(index=False))
    print("\n[최근 8분기 (억원)]")
    print((wide.tail(8).T / 1e8).round(0).to_string())
    return wide, mp


def show_dart_is_accounts(ticker: str, sj: Tuple[str, ...] = ("IS", "CIS"), db_info: dict = None) -> pd.DataFrame:
    """매핑 실패 진단: 해당 종목의 계정 목록 — account_id / account_nm / 등장 횟수."""
    ticker = str(ticker).upper().lstrip("A").zfill(6)
    df = load_dart_long(ticker, db_info or get_db_info())
    out = (df[df["sj_div"].isin(list(sj))].groupby(["sj_div", "account_id", "account_nm"]).size()
           .reset_index(name="n").sort_values(["sj_div", "n"], ascending=[True, False]))
    print(out.to_string(index=False)); return out

## PART 2 · 매출 예측 (Korea_revenue_forecast_v4 준용) → `korea_revenue_forecast_from_DART`

- `_prepare_series` / `_run_models` 는 v4 와 동일 (최근 연속구간 추출, 전 모형 실패 차단, median 앙상블, n_models)
- 원천만 DART wide 의 `revenue` 로 교체 (단위 원)
- **저장 테이블 신규** — `forecast_date`(계측일) 가 UNIQUE KEY 에 포함되어 **계측일별 vintage** 가 누적됨 (요구사항 4)

In [ ]:
def get_dart_revenue_series(ticker: str, db_info: dict, verbose: bool = False) -> Tuple[pd.Series, pd.DataFrame]:
    """DART wide → 분기 매출 PeriodIndex Series (원). v4 _prepare_series 와 동일 규칙."""
    wide, _ = load_dart_financials_wide(ticker, db_info, verbose=verbose)
    if wide.empty or "revenue" not in wide.columns:
        return pd.Series(dtype=float), wide
    rev = wide["revenue"].dropna()
    rev = rev[rev != 0]
    if rev.empty:
        return pd.Series(dtype=float), wide
    s = rev.copy()
    s.index = pd.PeriodIndex(s.index, freq="Q")
    s = s[~s.index.duplicated(keep="last")].sort_index().astype(float)

    # ── 최근 연속 구간 추출 (v4) ──
    full = pd.period_range(s.index.min(), s.index.max(), freq="Q")
    if len(full) != len(s):
        missing = full.difference(s.index)
        n_before = len(s)
        s = s.loc[s.index > missing.max()]
        if verbose:
            log(ticker, f"[결측분기 {len(missing)}개] {n_before}Q → 최근 연속 {len(s)}Q (마지막 결측 {missing.max()})")
    return s, wide


def _run_models(series: pd.Series, forecast_quarters: int) -> Tuple[pd.DataFrame, dict]:
    """v4 동일: SARIMA/ETS/Theta → wide DataFrame(index=예측 Period) + raw dict."""
    m = 4
    raw = {
        "SARIMA": forecast_sarima(y=series, forecast_horizon=forecast_quarters, seasonal_period=m, try_transforms=True),
        "ETS":    forecast_ets(y=series, forecast_horizon=forecast_quarters, m=m, try_transforms=True),
        "Theta":  forecast_theta(y=series, forecast_horizon=forecast_quarters, m=m, try_transforms=True),
    }
    ok = [k for k, r in raw.items() if isinstance(r, dict) and "forecast" in r]
    if not ok:
        errs = "; ".join(f"{k}:{(r or {}).get('error', 'unknown')}" for k, r in raw.items())
        raise ValueError(f"전 모형 적합 실패 ({errs})")
    periods = pd.period_range(start=series.index[-1] + 1, periods=forecast_quarters, freq="Q")
    fc = pd.DataFrame(index=periods)
    for name in ("SARIMA", "ETS", "Theta"):
        vals = raw[name].get("forecast") if isinstance(raw[name], dict) else None
        fc[name] = np.asarray(vals, dtype=float) if vals is not None else np.nan
    cols = ["SARIMA", "ETS", "Theta"]
    fc["Ensemble"] = fc[cols].median(axis=1) if ENSEMBLE_AGG == "median" else fc[cols].mean(axis=1)
    fc["n_models"] = fc[cols].notna().sum(axis=1)
    return fc, raw


def forecast_dart_revenue(ticker: str, db_info: dict,
                          forecast_quarters: int = FORECAST_QUARTERS,
                          min_data_periods: int = MIN_DATA_PERIODS,
                          verbose: bool = False) -> Tuple[bool, pd.DataFrame, str]:
    """
    단일 종목 예측. Returns (성공, wide[date,SARIMA,ETS,Theta,Ensemble,n_models,ticker,last_actual_date,n_obs], msg)
    """
    try:
        series, _ = get_dart_revenue_series(ticker, db_info, verbose=verbose)
        if series.empty:
            return False, pd.DataFrame(), "DART 매출 없음"
        if len(series) < min_data_periods:
            return False, pd.DataFrame(), f"데이터 부족 ({len(series)}Q < {min_data_periods}Q)"
        fc, _ = _run_models(series, forecast_quarters)
        fc["ticker"] = ticker
        fc["last_actual_date"] = series.index[-1].to_timestamp(how="end").normalize().date()
        fc["n_obs"] = len(series)
        fc.index = fc.index.to_timestamp(how="end").normalize()
        fc = fc.reset_index().rename(columns={"index": "date"})
        return True, fc, ""
    except Exception as e:
        if verbose:
            log(ticker, f"예측 실패 - {e}")
        return False, pd.DataFrame(), str(e)


# ── DB 저장 (vintage) ──────────────────────────────────────────
CREATE_DART_FC_SQL = f"""
CREATE TABLE IF NOT EXISTS {TABLE_DART_FC} (
    id               BIGINT AUTO_INCREMENT PRIMARY KEY,
    forecast_date    DATE        NOT NULL COMMENT '★ 계측일 (vintage key)',
    ticker           VARCHAR(20) NOT NULL COMMENT 'DART 6자리',
    date             DATE        NOT NULL COMMENT '예측 대상 분기말',
    indicator        VARCHAR(50) NOT NULL COMMENT 'SARIMA|ETS|Theta|Ensemble',
    value            DOUBLE               COMMENT '매출 (원)',
    n_models         TINYINT              COMMENT '앙상블 구성 모형 수',
    last_actual_date DATE                 COMMENT '예측 입력 마지막 실적 분기',
    n_obs            INT                  COMMENT '입력 분기 수',
    unit             VARCHAR(10) DEFAULT 'KRW',
    source           VARCHAR(40) DEFAULT '{TABLE_DART_FS}',
    ensemble_agg     VARCHAR(10),
    fc_module        VARCHAR(60),
    created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    UNIQUE KEY uq_vintage (forecast_date, ticker, date, indicator),
    INDEX idx_ticker (ticker), INDEX idx_fdate (forecast_date), INDEX idx_date (date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='DART 기반 분기 매출 예측 (계측일별 vintage)'
"""

def ensure_dart_fc_table(db_info: dict):
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(CREATE_DART_FC_SQL)
        conn.commit()
    finally:
        conn.close()


def convert_to_long_format(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["date", "ticker", "indicator", "value", "n_models", "last_actual_date", "n_obs"])
    long_df = df.melt(id_vars=["date", "ticker", "n_models", "last_actual_date", "n_obs"],
                      value_vars=["SARIMA", "ETS", "Theta", "Ensemble"],
                      var_name="indicator", value_name="value")
    return long_df.sort_values(["ticker", "date", "indicator"]).reset_index(drop=True)


def save_dart_forecasts(forecasts_long: pd.DataFrame, db_info: dict,
                        forecast_date: date = FORECAST_DATE, batch_size: int = 200) -> int:
    """UNIQUE(forecast_date, ticker, date, indicator) → 같은 계측일 재실행 시 value 갱신."""
    if forecasts_long.empty:
        print("저장할 데이터 없음"); return 0
    ensure_dart_fc_table(db_info)
    sql = f"""
        INSERT INTO {TABLE_DART_FC}
            (forecast_date, ticker, date, indicator, value, n_models, last_actual_date, n_obs,
             unit, source, ensemble_agg, fc_module)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,'KRW',%s,%s,%s)
        ON DUPLICATE KEY UPDATE value=VALUES(value), n_models=VALUES(n_models),
            last_actual_date=VALUES(last_actual_date), n_obs=VALUES(n_obs),
            ensemble_agg=VALUES(ensemble_agg), fc_module=VALUES(fc_module), updated_at=CURRENT_TIMESTAMP
    """
    rows = [(forecast_date, r.ticker, pd.Timestamp(r.date).date(), r.indicator,
             float(r.value) if pd.notna(r.value) else None, int(r.n_models),
             r.last_actual_date, int(r.n_obs), TABLE_DART_FS, ENSEMBLE_AGG, FORECAST_MODULE)
            for r in forecasts_long.itertuples(index=False)]
    conn = get_pymysql_conn(db_info)
    n = 0
    try:
        with conn.cursor() as cur:
            for i in range(0, len(rows), batch_size):
                cur.executemany(sql, rows[i:i + batch_size])
                n += len(rows[i:i + batch_size])
        conn.commit()
    except Exception:
        conn.rollback(); raise
    finally:
        conn.close()
    print(f"  저장 완료 | {n:,}행 → {TABLE_DART_FC}  (forecast_date={forecast_date})")
    return n


def forecast_exists(ticker: str, db_info: dict, forecast_date: date) -> bool:
    ensure_dart_fc_table(db_info)
    with engine.connect() as c:
        r = c.execute(_sa_text(f"SELECT COUNT(*) FROM {TABLE_DART_FC} WHERE ticker=:t AND forecast_date=:d"),
                      {"t": ticker, "d": forecast_date}).fetchone()
    return bool(r and r[0] > 0)


def run_dart_forecast_batch(tickers: List[str], db_info: dict, forecast_date: date = FORECAST_DATE,
                            rerun: bool = RERUN_FORECAST, verbose: bool = False) -> Dict[str, Any]:
    """다수 종목 예측 + 저장. (v4 process_all_tickers 축약판)"""
    ok, fail, buf = 0, [], []
    t0 = time.time()
    for tk in tqdm(tickers, desc="DART 매출예측"):
        if not rerun and forecast_exists(tk, db_info, forecast_date):
            if verbose: log(tk, f"forecast_date={forecast_date} 이미 존재 → skip")
            ok += 1; continue
        s, df, msg = forecast_dart_revenue(tk, db_info, verbose=verbose)
        if s:
            ok += 1; buf.append(df)
        else:
            fail.append((tk, msg))
        gc.collect()
    if buf:
        save_dart_forecasts(convert_to_long_format(pd.concat(buf, ignore_index=True)), db_info, forecast_date)
    print(f"[예측 완료] 성공 {ok} / 실패 {len(fail)}  경과 {time.time()-t0:.0f}s")
    for tk, m in fail:
        print(f"  ✗ {tk}: {m}")
    return {"success": ok, "fail": fail}


# ── 조회: 최신(또는 지정) vintage 로드 + v1 3중 검증 ──────────
_FC_MODEL_PRIORITY = ("Ensemble", "SARIMA", "ETS", "Theta")

def load_dart_revenue_forecast(ticker: str, db_info: dict, horizon: int = FORECAST_HORIZON,
                               forecast_date: Optional[date] = None) -> Tuple[pd.Series, str, Optional[str]]:
    """
    Returns (forecast Series[원, index=분기말], indicator, forecast_date str)
    forecast_date=None → 가장 최근 계측일.  v1 패치의 3중 검증(분기수·연속성·상수) 동일 적용.
    """
    ensure_dart_fc_table(db_info)
    with engine.connect() as c:
        if forecast_date is None:
            r = c.execute(_sa_text(f"SELECT MAX(forecast_date) FROM {TABLE_DART_FC} WHERE ticker=:t"), {"t": ticker}).fetchone()
            if r is None or r[0] is None:
                return pd.Series(dtype=float), "", None
            forecast_date = r[0]
    fd = str(forecast_date)
    for model in _FC_MODEL_PRIORITY:
        df = pd.read_sql(_sa_text(f"SELECT date, value FROM {TABLE_DART_FC} "
                                  f"WHERE ticker=:t AND forecast_date=:d AND indicator=:m ORDER BY date"),
                         engine, params={"t": ticker, "d": fd, "m": model})
        df = df.dropna(subset=["value"])
        if not df.empty:
            break
    else:
        return pd.Series(dtype=float), "", fd
    fc = df.assign(date=pd.to_datetime(df["date"])).set_index("date")["value"].astype(float).sort_index()
    if len(fc) < horizon:
        raise ValueError(f"[{ticker}] forecast {len(fc)}Q < horizon {horizon} (vintage {fd}) → 재예측 필요")
    fc = fc.iloc[:horizon]
    per = fc.index.to_period("Q")
    if ((per[1:].astype("int64") - per[:-1].astype("int64")) != 1).any():
        raise ValueError(f"[{ticker}] forecast 분기 불연속 (vintage {fd})")
    if fc.nunique() == 1:
        raise ValueError(f"[{ticker}] forecast 전 분기 동일값 — 상수 시계열 의심 (vintage {fd})")
    return fc, model, fd


def load_forecast_vintages(ticker: str, db_info: dict, indicator: str = "Ensemble") -> pd.DataFrame:
    """계측일(forecast_date) × 대상분기(date) 피벗 — 예측이 계측일별로 어떻게 변했는지 추적 (요구사항 4)."""
    ensure_dart_fc_table(db_info)
    df = pd.read_sql(_sa_text(f"SELECT forecast_date, date, value, last_actual_date FROM {TABLE_DART_FC} "
                              f"WHERE ticker=:t AND indicator=:m ORDER BY forecast_date, date"),
                     engine, params={"t": ticker, "m": indicator})
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"]).dt.to_period("Q").astype(str)
    pv = df.pivot_table(index="forecast_date", columns="date", values="value", aggfunc="last")
    last = df.groupby("forecast_date")["last_actual_date"].first()
    pv.insert(0, "last_actual", last.astype(str))
    return pv


def inspect_dart_ticker(ticker: str, db_info: dict = None, tail_n: int = 20, forecast_quarters: int = FORECAST_QUARTERS) -> dict:
    """개별 종목 점검 (DB 저장 X): 입력 매출 tail + 모델별 예측 + 적합 상태 (v4 inspect_ticker 준용)."""
    db_info = db_info or get_db_info()
    ticker = str(ticker).upper().lstrip("A").zfill(6)
    series, wide = get_dart_revenue_series(ticker, db_info, verbose=True)
    if series.empty:
        print(f"[{ticker}] DART 매출 없음"); return {}
    fc, raw = _run_models(series, forecast_quarters)
    tail = (series.tail(tail_n) / 1e8).rename("revenue(억원)").to_frame(); tail.index = tail.index.astype(str)
    print("=" * 70); print(f" {ticker} | 사용 {len(series)}Q | 예측 {forecast_quarters}Q | 앙상블 {ENSEMBLE_AGG}"); print("=" * 70)
    print(tail.to_string(float_format=lambda x: f"{x:,.0f}"))
    print("\n[모델 적합 상태]")
    for nm in ("SARIMA", "ETS", "Theta"):
        r = raw.get(nm, {}) or {}
        if "forecast" in r:
            spec = r.get("spec", {}) or {}
            extra = f"  order={spec.get('order')} seasonal={spec.get('seasonal_order')}" if nm == "SARIMA" else ""
            print(f"   {nm:<7} 성공   transform={','.join(r.get('used_transform', [])) or 'none'}{extra}")
        else:
            print(f"   {nm:<7} 실패   {r.get('error', 'unknown')}")
    disp = (fc[["SARIMA", "ETS", "Theta", "Ensemble"]] / 1e8).round(0); disp["n_models"] = fc["n_models"]; disp.index = disp.index.astype(str)
    print("\n[모델별 예측 (억원)]"); print(disp.to_string(float_format=lambda x: f"{x:,.0f}"))
    return {"ticker": ticker, "input_series": series, "forecast": fc, "raw": raw, "wide": wide}

## PART 3 · `DartDCFModel` — KoreaDCFModel v8 (US v13 가드) 그대로, 데이터 로더만 DART

변경점은 `load_sales` / `load_financials` / `_estimate_shares` 세 곳뿐. 나머지(OPM 앙상블, 비율계수 OLS/median, NWC v7,
WACC 가드, EVA→Moat→ρ, Phase2 AR(1), TV 스프레드·배수 상한, Sanity Guard)는 `korea_fcff_individual_valuation_v1` 원문과 동일.

**DART 로 추정되는 부족 항목 (요구사항 5)**

| 항목 | 추정 방식 |
|---|---|
| 주식수 | `시가총액(백만원)×1e6 ÷ 현재가` (DART FS API 에 주식수 없음) |
| 이자비용 | `이자비용` 계정 없으면 `금융비용` |
| CapEx | CF 취득 계정 없으면 `ΔPPE + D&A` |
| 영업운전자본 부채항목 | 미지급비용·미지급금·선수금·계약부채 중 하나라도 있으면 v7 방식, 없으면 legacy proxy |

In [ ]:
class DartDCFModel:
    """DART 원천 Sales-driven FCFF DCF (KoreaDCFModel v8 이식)."""

    NWC_FIELD_ALIASES = {
        "receivables": ["receivables"], "inventories": ["inventories"],
        "short_term_debt": ["short_term_debt"], "current_lt_debt": ["current_lt_debt"], "lease_liab": ["lease_liab"],
    }
    NWC_OP_ASSET_ALIASES = {"receivables": ["receivables"], "inventories": ["inventories"], "prepaid_expenses": ["prepaid_expenses"]}
    NWC_OP_LIAB_ALIASES = {"payables": ["payables"], "accrued_expenses": ["accrued_expenses"], "other_payables": ["other_payables"],
                           "advances_received": ["advances_received"], "contract_liabilities": ["contract_liabilities"]}

    def __init__(self, ticker, engine, db_info, rf, e_rm, kospi_series,
                 forecast_horizon=FORECAST_HORIZON, min_history=MIN_HISTORY, gdp_growth=GDP_GROWTH,
                 forecast_date: Optional[date] = None, verbose=False):
        self.ticker_dart  = str(ticker).upper().lstrip("A").zfill(6)
        self.ticker_dg    = to_dg_ticker(self.ticker_dart)       # 시총/베타 helper 용 (A005930)
        self.ticker_price = to_price_ticker(self.ticker_dart)
        self.engine, self.db_info = engine, db_info
        self.rf, self.e_rm, self.erp = rf, e_rm, e_rm - rf
        self.kospi = kospi_series
        self.horizon, self.min_history, self.gdp_growth = forecast_horizon, min_history, gdp_growth
        self.forecast_date_req = forecast_date
        self.verbose = verbose

        self._sales_actual: Optional[pd.Series] = None
        self._sales_forecast: Optional[pd.Series] = None
        self._used_model = ""; self._forecast_date = None
        self._fs_wide: Optional[pd.DataFrame] = None
        self._mapping_df: Optional[pd.DataFrame] = None
        self._fcff_history: Optional[pd.Series] = None
        self._beta_info = None; self._mkt_cap = None; self._current_price = None
        self._last_actual_nwc = None; self._nwc_method = "?"
        self._op_ca_latest = np.nan; self._op_cl_latest = np.nan; self._nwc_to_sales = np.nan
        self._shares_method = "?"
        self._eva_cache = {}
        self.result_df: Optional[pd.DataFrame] = None
        self.valuation: Optional[Dict] = None
        self.report = DataQualityReport(ticker=self.ticker_dart)

    # ── 1. 데이터 로드 (★ DART) ──────────────────────────────
    def load_sales(self):
        series, wide = get_dart_revenue_series(self.ticker_dart, self.db_info, verbose=self.verbose)
        if series.empty:
            self.report.add("revenue_actual", "missing", n_obs=0, note="DART 매출 없음")
            raise ValueError(f"[{self.ticker_dart}] DART Sales actual 없음")
        actual = series.copy(); actual.index = actual.index.to_timestamp(how="end").normalize()
        actual = actual * FS_UNIT_MULTIPLIER
        if len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual), note=f"매출 {len(actual)}Q < {MIN_REVENUE_QUARTERS}")
            raise ValueError(f"[{self.ticker_dart}] 매출 {len(actual)}Q < 최소 {MIN_REVENUE_QUARTERS}Q")
        self.report.add("revenue_actual", "ok", n_obs=len(actual))

        forecast, model_name, fd = load_dart_revenue_forecast(self.ticker_dart, self.db_info, horizon=self.horizon,
                                                              forecast_date=self.forecast_date_req)
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0, note=f"{TABLE_DART_FC} 에 예측 없음")
            raise ValueError(f"[{self.ticker_dart}] 매출 forecast 없음 — PART 2 예측 먼저 실행")
        forecast = forecast * FS_UNIT_MULTIPLIER
        # 예측 시작 분기가 실적 다음 분기인지 확인 (오래된 vintage 방지)
        exp_start = (actual.index[-1].to_period("Q") + 1)
        if forecast.index[0].to_period("Q") != exp_start:
            self.report.warn(f"forecast 시작 {forecast.index[0].to_period('Q')} ≠ 실적 다음 분기 {exp_start} (vintage {fd})")
            if self.verbose: log(self.ticker_dart, f"⚠️ forecast 시작분기 불일치 — 최신 실적 반영 재예측 권장")
        self.report.add("revenue_forecast", "ok", n_obs=len(forecast), note=f"model={model_name}, forecast_date={fd}")
        self._sales_actual, self._sales_forecast = actual, forecast.iloc[:self.horizon]
        self._used_model, self._forecast_date = model_name, fd
        if self.verbose:
            log(self.ticker_dart, f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q model={model_name} forecast_date={fd}")
        return self

    def load_financials(self):
        wide, mp = load_dart_financials_wide(self.ticker_dart, self.db_info, verbose=self.verbose)
        if wide.empty:
            raise ValueError(f"[{self.ticker_dart}] DART wide 비어있음")
        wide = wide * FS_UNIT_MULTIPLIER
        if "operating_income" not in wide.columns or wide["operating_income"].dropna().empty:
            self.report.add("operating_income", "missing", n_obs=0, note="영업이익 없음")
            raise ValueError(f"[{self.ticker_dart}] operating_income 없음")
        self.report.add("operating_income", "ok", n_obs=int(wide["operating_income"].notna().sum()))
        for std_name, aliases in self.NWC_FIELD_ALIASES.items():
            present = [a for a in aliases if a in wide.columns]
            if not present:
                if self.verbose: log(self.ticker_dart, f"⚠️  NWC 필드 없음: {std_name}")
            elif pd.to_numeric(wide[present[0]], errors="coerce").abs().sum() == 0:
                if self.verbose: log(self.ticker_dart, f"⚠️  NWC 필드 전부 0: {present[0]}")
        self._fs_wide, self._mapping_df = wide, mp
        return self

    # ── 2. 변수별 추정 (v8 원문) ─────────────────────────────
    @staticmethod
    def _winsorize(s, limits=WINSORIZE_LIMITS):
        s = s.dropna()
        if len(s) < 4: return s
        return s.clip(s.quantile(limits[0]), s.quantile(limits[1]))

    @staticmethod
    def _ols_ratio(x, y):
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES: return np.nan, -1.0, int(mask.sum())
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return float(slope), float(r ** 2), int(mask.sum())

    def _resolve_alias(self, std_name):
        for a in self.NWC_FIELD_ALIASES.get(std_name, [std_name]):
            if a in self._fs_wide.columns: return a
        return None

    def _resolve_from(self, alias_map, std_name):
        for a in alias_map.get(std_name, [std_name]):
            if a in self._fs_wide.columns: return a
        return None

    def _sum_components(self, alias_map):
        wide = self._fs_wide; total = pd.Series(0.0, index=wide.index); resolved = []
        for std_name in alias_map:
            col = self._resolve_from(alias_map, std_name)
            if col is not None:
                total = total + pd.to_numeric(wide[col], errors="coerce").fillna(0); resolved.append(std_name)
        return total, resolved

    def estimate_tax_rate(self):
        if "pretax_income" not in self._fs_wide.columns or "tax_expense" not in self._fs_wide.columns:
            self.report.add("tax_rate", "fallback_zero", n_obs=0, value=0.22, note="세전이익/법인세 없음 → 22%")
            return 0.22
        df = self._fs_wide[["pretax_income", "tax_expense"]].dropna(); df = df[df["pretax_income"] > 0]
        if df.empty:
            self.report.add("tax_rate", "fallback_zero", n_obs=0, value=0.22, note="법정세율 22% fallback"); return 0.22
        rates = (df["tax_expense"] / df["pretax_income"]).clip(0, 0.40); med = float(rates.median())
        self.report.add("tax_rate", "ok", n_obs=len(rates), value=med); return med

    def estimate_opm(self, sales_series):
        df = self._fs_wide[["operating_income", "revenue"]].dropna(); df = df[df["revenue"] > 0]
        df["opm"] = self._winsorize(df["operating_income"] / df["revenue"]); opm_series = df["opm"].dropna()
        if len(opm_series) < self.min_history:
            fb = float(opm_series.median()) if not opm_series.empty else 0.05
            self.report.add("opm_forecast", "fallback_median", n_obs=len(opm_series), value=fb, note=f"OPM history {len(opm_series)} < {self.min_history}")
            return pd.Series([fb] * len(sales_series), index=sales_series.index)
        m = 4
        forecasts = {}
        for name, fn in [("SARIMA", lambda y: forecast_sarima(y, self.horizon, seasonal_period=m)),
                         ("ETS", lambda y: forecast_ets(y, self.horizon, m=m)),
                         ("Theta", lambda y: forecast_theta(y, self.horizon, m=m))]:
            try:
                res = fn(opm_series)
                if "forecast" in res and "error" not in res: forecasts[name] = np.asarray(res["forecast"])
            except Exception:
                pass
        if forecasts:
            ens = np.clip(np.mean(list(forecasts.values()), axis=0), -0.30, 0.50)
            self.report.add("opm_forecast", "ok", n_obs=len(opm_series), note=f"models={list(forecasts.keys())}")
            return pd.Series(ens, index=sales_series.index)
        fb = float(np.clip(opm_series.tail(8).median() if len(opm_series) >= 8 else opm_series.median(), -0.30, 0.50))
        self.report.add("opm_forecast", "fallback_median", n_obs=len(opm_series), value=fb, note="모든 ts 모델 실패")
        return pd.Series([fb] * len(sales_series), index=sales_series.index)

    def estimate_ratio_coef(self, target_keys, ratio_name, take_abs=False):
        sales = self._fs_wide["revenue"].dropna(); sales = sales[sales > 0]
        if sales.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0, note="revenue 없음"); return 0.0, "missing"
        target = pd.Series(0.0, index=self._fs_wide.index); valid_cols = []
        for k in target_keys:
            if k in self._fs_wide.columns:
                target = target + self._fs_wide[k].fillna(0.0); valid_cols.append(k)
        if not valid_cols:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0, note=f"{target_keys} 모두 없음 (DART 미제공)"); return 0.0, "missing"
        if take_abs: target = target.abs()
        merged = pd.concat([sales.rename("rev"), target.rename("y")], axis=1, join="inner").dropna(); merged = merged[merged["rev"] > 0]
        if merged.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0, note="merged 비어있음"); return 0.0, "empty"
        slope, r2, n = self._ols_ratio(merged["rev"], merged["y"])
        if (not np.isnan(slope) and r2 >= OLS_MIN_R2 and slope >= 0 and n >= OLS_MIN_SAMPLES):
            self.report.add(ratio_name, "ok", n_obs=n, value=slope, r2=r2, note=f"OLS cols={valid_cols}"); return float(slope), "ols"
        ratios = (merged["y"] / merged["rev"]).replace([np.inf, -np.inf], np.nan).dropna()
        if ratios.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0); return 0.0, "ratio_empty"
        med = float(self._winsorize(ratios).median())
        self.report.add(ratio_name, "fallback_median", n_obs=len(ratios), value=med, r2=r2, note=f"OLS R²={r2:.2f} → median, cols={valid_cols}")
        return max(med, 0.0), "median_ratio"

    def _compute_operating_nwc_series(self):
        wide = self._fs_wide
        op_ca, ca_res = self._sum_components(self.NWC_OP_ASSET_ALIASES)
        op_cl, cl_res = self._sum_components(self.NWC_OP_LIAB_ALIASES)
        nwc = op_ca - op_cl
        self._op_ca_latest = float(op_ca.iloc[-1]) if len(op_ca) else np.nan
        self._op_cl_latest = float(op_cl.iloc[-1]) if len(op_cl) else np.nan
        return pd.DataFrame({"date": wide.index, "nwc": nwc.values}).reset_index(drop=True), ca_res, cl_res, len(cl_res) > 0

    def _compute_legacy_nwc_series(self):
        wide = self._fs_wide
        rc, ic = self._resolve_alias("receivables"), self._resolve_alias("inventories")
        rec = pd.to_numeric(wide[rc], errors="coerce").fillna(0) if rc else pd.Series(0.0, index=wide.index)
        inv = pd.to_numeric(wide[ic], errors="coerce").fillna(0) if ic else pd.Series(0.0, index=wide.index)
        st = pd.Series(0.0, index=wide.index)
        for s in ["short_term_debt", "current_lt_debt", "lease_liab"]:
            col = self._resolve_alias(s)
            if col: st = st + pd.to_numeric(wide[col], errors="coerce").fillna(0)
        self._op_ca_latest = float((rec + inv).iloc[-1]) if len(wide) else np.nan
        self._op_cl_latest = float(st.iloc[-1]) if len(wide) else np.nan
        return pd.DataFrame({"date": wide.index, "nwc": ((rec + inv) - st).values}).reset_index(drop=True)

    def _compute_bs_nwc_series(self):
        if USE_OPERATING_NWC:
            df, ca_res, cl_res, has = self._compute_operating_nwc_series()
            if has:
                self._nwc_method = "operating_v7"
                self.report.add("nwc_definition", "ok", n_obs=len(df), note=f"operating_v7 CA={ca_res} CL={cl_res}")
                if self.verbose:
                    log(self.ticker_dart, f"NWC[v7] CA={ca_res} CL={cl_res}  최근 NWC={(self._op_ca_latest-self._op_cl_latest)/1e9:,.1f}B")
                return df
            self.report.warn("v7 영업부채 항목 없음 → legacy NWC proxy")
        self._nwc_method = "legacy_proxy"
        return self._compute_legacy_nwc_series()

    def estimate_nwc_coef(self):
        nwc_df = self._compute_bs_nwc_series()
        self._last_actual_nwc = float(nwc_df["nwc"].iloc[-1]) if not nwc_df.empty else 0.0
        sales = self._fs_wide["revenue"]
        merged = pd.DataFrame({"date": sales.index, "rev": sales.values}).merge(nwc_df, on="date", how="inner").dropna()
        merged = merged[merged["rev"] > 0]
        if merged.empty:
            self.report.add("nwc", "fallback_zero", n_obs=0, value=0.0, note="merged 비어있음"); return 0.0, "empty"
        slope, r2, n = self._ols_ratio(merged["rev"], merged["nwc"])
        if (not np.isnan(slope) and r2 >= OLS_MIN_R2 and n >= OLS_MIN_SAMPLES):
            self.report.add("nwc", "ok", n_obs=n, value=slope, r2=r2); return float(slope), "ols"
        ratios = (merged["nwc"] / merged["rev"]).replace([np.inf, -np.inf], np.nan).dropna()
        if ratios.empty:
            self.report.add("nwc", "fallback_zero", n_obs=0, value=0.0); return 0.0, "ratio_empty"
        med = float(self._winsorize(ratios).median())
        self.report.add("nwc", "fallback_median", n_obs=len(ratios), value=med, r2=r2, note=f"OLS R²={r2:.2f} → median"); return med, "median_ratio"

    # ── 3. FCFF ──────────────────────────────────────────────
    def compute_fcff(self):
        sales_fc = self._sales_forecast; tax = self.estimate_tax_rate(); opm_fc = self.estimate_opm(sales_fc)
        alpha, _ = self.estimate_ratio_coef(["da_cf", "intangible_amort_cf"], "da")
        beta_, _ = self.estimate_ratio_coef(["capex_tangible", "capex_intangible"], "capex", take_abs=True)
        gamma, _ = self.estimate_nwc_coef(); self._nwc_to_sales = gamma
        last_actual_sales = float(self._sales_actual.iloc[-1])
        prev_nwc = gamma * last_actual_sales
        if self.verbose and self._last_actual_nwc is not None and abs(prev_nwc) > 0:
            gap = abs(self._last_actual_nwc - prev_nwc) / max(abs(self._last_actual_nwc), 1)
            if gap > 0.5:
                log(self.ticker_dart, f"[NWC seed] γ×sales={prev_nwc/1e9:,.1f}B vs 실측={self._last_actual_nwc/1e9:,.1f}B 괴리 {gap:.1%}")
        rows = []
        for i, (dt, sales) in enumerate(sales_fc.items()):
            opm = float(opm_fc.iloc[i]); ebit = sales * opm; nopat = ebit * (1 - tax)
            da, capex, nwc = alpha * sales, beta_ * sales, gamma * sales
            dnwc = nwc - prev_nwc; prev_nwc = nwc
            fcff = nopat + da - capex - dnwc
            invested = capex + max(dnwc, 0)
            rows.append({"date": dt, "quarter": f"{dt.year}Q{dt.quarter}", "sales_forecast": sales, "opm_forecast": opm,
                         "ebit": ebit, "tax_rate": tax, "nopat": nopat, "da": da, "capex": capex, "nwc": nwc,
                         "delta_nwc": dnwc, "fcff": fcff,
                         "roic": nopat / invested if invested > 1e-6 else np.nan,
                         "reinvestment_rate": capex / nopat if nopat > 1e-6 else np.nan})
        self.result_df = pd.DataFrame(rows)
        self._coefs = {"tax": tax, "alpha_da": alpha, "beta_capex": beta_, "gamma_nwc": gamma}
        self._compute_historical_fcff(tax, alpha, beta_, gamma)
        return self

    def _compute_historical_fcff(self, tax, alpha, beta_, gamma):
        sales_act, wide, rows = self._sales_actual, self._fs_wide, []
        prev_nwc = gamma * float(sales_act.iloc[0])
        for dt, sales in sales_act.items():
            if dt not in wide.index: continue
            oi, rv = wide.loc[dt].get("operating_income", np.nan), wide.loc[dt].get("revenue", np.nan)
            if pd.isna(oi) or pd.isna(rv) or rv <= 0:
                prev_nwc = gamma * sales; continue
            nopat = sales * (float(oi) / float(rv)) * (1 - tax)
            nwc = gamma * sales; dnwc = nwc - prev_nwc; prev_nwc = nwc
            rows.append({"date": dt, "fcff": nopat + alpha * sales - beta_ * sales - dnwc})
        self._fcff_history = pd.DataFrame(rows).set_index("date")["fcff"] if rows else pd.Series(dtype=float)

    # ── 4. WACC ──────────────────────────────────────────────
    def compute_beta_re(self):
        info = compute_beta_10y(self.ticker_dg, self.db_info, kospi_series=self.kospi, years=10, min_obs=750)
        self._beta_info = info
        if np.isnan(info["beta_raw"]):
            bb = 1.0; self.report.add("beta", "fallback_median", n_obs=info["n_obs"], value=bb, note="베타 실패 → 1.0")
        else:
            bb = info["beta_blume"]; self.report.add("beta", "ok", n_obs=info["n_obs"], value=bb, r2=info["r_squared"], note=f"β_raw={info['beta_raw']:.3f}")
        re = self.rf + bb * self.erp
        if self.verbose: log(self.ticker_dart, f"β_raw={info.get('beta_raw', np.nan):.3f} β_blume={bb:.3f} Re={re:.3%} (n={info['n_obs']})")
        return float(re), bb

    def _total_debt_series(self):
        wide = self._fs_wide; td = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns: td = td + wide[k].fillna(0)
        return td

    def compute_cost_of_debt(self, tax):
        wide = self._fs_wide
        if "interest_expense" not in wide.columns or wide["interest_expense"].dropna().empty:
            self.report.add("rd", "fallback_zero", value=RD_DEFAULT, note="이자비용·금융비용 모두 없음 → RD_DEFAULT"); return RD_DEFAULT
        td = self._total_debt_series(); td_avg = (td + td.shift(1)) / 2
        ie = wide["interest_expense"].abs(); valid = (td_avg > 0) & ie.notna()
        if valid.sum() < 4:
            self.report.add("rd", "fallback_zero", n_obs=int(valid.sum()), value=RD_DEFAULT, note="유효 분기 < 4"); return RD_DEFAULT
        mult = 4.0 if RD_ANNUALIZE_QUARTERLY else 1.0
        rd = float((ie[valid] * mult / td_avg[valid]).clip(0, 0.20).median())
        if V8_RD_FLOOR_RF and rd < self.rf:
            self.report.add("rd", "ok", n_obs=int(valid.sum()), value=float(self.rf), note=f"median {rd:.3%} < Rf → Rf floor"); return float(self.rf)
        self.report.add("rd", "ok", n_obs=int(valid.sum()), value=rd); return rd

    def _estimate_shares(self) -> float:
        """★ DART 에 주식수 없음 → 시가총액 ÷ 현재가 추정."""
        mc, mc_date = load_korea_marketcap_latest(self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        if mc and mc > 0 and cp and cp > 0:
            sh = mc * MARKETCAP_UNIT_MULTIPLIER / cp; self._shares_method = f"marketcap/price ({mc_date})"
            self.report.add("shares", "ok", value=sh, note=self._shares_method); return float(sh)
        self._shares_method = "missing"; self.report.add("shares", "missing", n_obs=0, note="시총 또는 주가 없음"); return np.nan

    def compute_wacc(self, re, tax):
        rd = self.compute_cost_of_debt(tax)
        mc, _ = load_korea_marketcap_latest(self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        if mc is None or mc <= 0:
            self.report.add("market_cap", "missing", n_obs=0, note="시가총액 없음 → Re 사용")
            if self.verbose: log(self.ticker_dart, "WACC: E 측정 실패 → Re만 사용")
            return float(re), re, rd
        E = mc * MARKETCAP_UNIT_MULTIPLIER
        td = self._total_debt_series().dropna(); D = float(td.sort_index().iloc[-1]) if not td.empty else 0.0
        V = E + D
        if V <= 0: return float(re), re, rd
        wacc = re * E / V + rd * (1 - tax) * D / V
        wacc = float(np.clip(wacc, max(V8_WACC_FLOOR_ABS, self.rf + 0.01), WACC_CAP))
        self._mkt_cap = E
        if self.verbose:
            log(self.ticker_dart, f"WACC={wacc:.3%} Re={re:.3%} Rd={rd:.3%} tax={tax:.2%} E/V={E/V:.1%} D/V={D/V:.1%} E={E/1e12:.2f}조 D={D/1e12:.2f}조")
        return wacc, re, rd

    # ── 5. EVA / Moat / Phase 2 (v8·v9.2 원문) ──────────────
    def _compute_eva_spread(self, wacc):
        wide = self._fs_wide
        empty = {"roic": np.nan, "eva_spread": np.nan, "ic": np.nan, "n_positive": 0, "eva_series": []}
        if "operating_income" not in wide.columns or "total_equity" not in wide.columns: return empty
        tax = self.estimate_tax_rate(); df = wide.copy(); df["nopat_q"] = df["operating_income"] * (1 - tax)
        cash = pd.Series(0.0, index=df.index)
        for k in CASH_KEYS:
            if k in df.columns: cash = cash + df[k].fillna(0)
        df["ic"] = df["total_equity"].fillna(0) + self._total_debt_series() - cash
        df = df[["nopat_q", "ic"]].dropna().sort_index()
        if len(df) < 4: return empty
        eva_series = []
        for i in range(3, len(df)):
            ic = float(df["ic"].iloc[i])
            if ic <= 0: continue
            roic = float(df["nopat_q"].iloc[i-3:i+1].sum()) / ic
            eva_series.append({"date": df.index[i], "roic": roic, "eva_spread": roic - wacc, "ic": ic})
        if not eva_series: return empty
        latest = eva_series[-1]; n_pos = sum(1 for e in eva_series[-20:] if e["eva_spread"] > 0)
        if self.verbose: log(self.ticker_dart, f"EVA: ROIC={latest['roic']:.2%} WACC={wacc:.2%} spread={latest['eva_spread']:+.2%} n_pos={n_pos}/20")
        return {"roic": latest["roic"], "eva_spread": latest["eva_spread"], "ic": latest["ic"], "n_positive": n_pos, "eva_series": eva_series}

    def _moat_to_rho_and_years(self, eva):
        if np.isnan(eva.get("eva_spread", np.nan)): return 0.75, 8, "Unknown (fallback)"
        s, n = eva["eva_spread"], eva["n_positive"]
        if   s > 0.15 and n >= 15: r = (0.90, 15, "Wide moat")
        elif s > 0.08 and n >= 12: r = (0.83, 12, "Narrow moat")
        elif s > 0.03 and n >= 8:  r = (0.75, 8, "Some moat")
        else:                      r = (0.60, 5, "No moat")
        if self.verbose: log(self.ticker_dart, f"Moat: [{r[2]}] spread={s:+.2%} n_pos={n}/20 → ρ={r[0]} Phase2={r[1]}yr")
        return r

    def _estimate_phase2_growth(self, wacc):
        G_TERM = self.gdp_growth; h = self._fcff_history
        g0_hist = g0_hist_method = None
        if h is not None and len(h.dropna()) >= 8:
            hc = h.dropna(); f0, fl = float(hc.iloc[-8]), float(hc.iloc[-1])
            if f0 > 0 and fl > 0:
                cagr_q = (fl / f0) ** (4.0 / 8) - 1
                g0_hist = float(np.clip((1 + cagr_q) ** 4 - 1, G_FLOOR_MIN, G_CEIL)); g0_hist_method = "8Q-CAGR"
        if g0_hist is None and h is not None and len(h.dropna()) >= 12:
            hc = h.dropna(); tr, tp = float(hc.iloc[-4:].sum()), float(hc.iloc[-8:-4].sum())
            if abs(tp) > 1e-6:
                if tp > 0: g0_hist = tr / tp - 1
                elif tp < 0 and tr < 0: g0_hist = (tr - tp) / abs(tp)
                else: g0_hist = G_TERM
                g0_hist = float(np.clip(g0_hist, G_FLOOR_MIN, G_CEIL)); g0_hist_method = "12Q-TTM"
        g0_fc = g0_fc_method = None
        if self.result_df is not None and len(self.result_df) >= 4:
            fq = self.result_df["fcff"].values; yr1 = float(np.sum(fq[:4])); yr2 = float(np.sum(fq[4:8])) if len(fq) >= 8 else yr1
            if abs(yr1) > 1e-6:
                if yr1 > 0: g0_fc = yr2 / yr1 - 1
                elif yr1 < 0 and yr2 < 0: g0_fc = (yr2 - yr1) / abs(yr1)
                else: g0_fc = G_TERM
                g0_fc = float(np.clip(g0_fc, G_FLOOR_MIN, G_CEIL)); g0_fc_method = "FC-yr1-yr2"
        if g0_hist is not None and g0_fc is not None:
            if g0_hist > 0 and g0_fc > 0:
                g0 = min(g0_hist, g0_fc); g0_method = f"min({g0_hist_method}={g0_hist:.1%}, FC={g0_fc:.1%})"
            elif g0_fc <= 0:
                g0 = g0_fc; g0_method = f"FC-priority (FC={g0_fc:.1%} ≤ 0, hist={g0_hist:.1%})"
            else:
                g0 = min(g0_hist, g0_fc); g0_method = f"min(hist={g0_hist:.1%}, FC={g0_fc:.1%}) 부호불일치"
        elif g0_hist is not None: g0, g0_method = g0_hist, g0_hist_method
        elif g0_fc is not None:   g0, g0_method = g0_fc, g0_fc_method
        else:                     g0, g0_method = G_TERM, "default-G_TERM"

        eva = self._compute_eva_spread(wacc); rho, n_years, moat_label = self._moat_to_rho_and_years(eva)
        if h is not None and len(h.dropna()) >= 12:
            hc = h.dropna(); ann = [float(hc.iloc[y:y+4].sum()) for y in range(0, len(hc) - len(hc) % 4, 4)]
            if len(ann) >= 4:
                gh = [(ann[i] / ann[i-1] - 1) if abs(ann[i-1]) > 1e-6 else 0.0 for i in range(1, len(ann))]
                if len(gh) >= 3:
                    yt = np.array([g - G_TERM for g in gh[1:]]); xt = np.array([g - G_TERM for g in gh[:-1]])
                    if np.dot(xt, xt) > 1e-10:
                        rho_ols = float(np.clip(np.dot(xt, yt) / np.dot(xt, xt), 0.40, 0.92))
                        rho = float(np.clip(0.5 * rho + 0.5 * rho_ols, 0.40, 0.92))
                        if self.verbose: log(self.ticker_dart, f"ρ 보정: OLS={rho_ols:.3f} → 혼합 {rho:.3f}")
        rho_cap_applied = False
        if V8_RHO_ADAPTIVE_CAP:
            cap = 0.75 if g0 > 0.30 else (0.85 if g0 > 0.15 else 0.92)
            if rho > cap:
                if self.verbose: log(self.ticker_dart, f"[v8] ρ 적응형 상한: g0={g0:.1%} ρ {rho:.3f}→{cap:.3f}")
                rho, rho_cap_applied = cap, True
        g, g_path = g0, []
        for _ in range(n_years):
            g = G_TERM + (g - G_TERM) * rho; g_path.append(float(np.clip(g, G_FLOOR_MIN, G_CEIL)))
        if self.verbose: log(self.ticker_dart, f"Phase2 [{moat_label}] ρ={rho:.3f} {n_years}년 g0={g0:.1%} ({g0_method}) → {g_path[0]:.1%} … {g_path[-1]:.1%}")
        self._eva_cache = {"eva_spread": eva.get("eva_spread", np.nan), "roic": eva.get("roic", np.nan), "n_positive": eva.get("n_positive", 0),
                           "moat_label": moat_label, "rho": rho, "n_phase2": n_years, "eva_series": eva.get("eva_series", []),
                           "g0": g0, "g0_method": g0_method, "g0_hist": g0_hist, "g0_fc": g0_fc, "rho_cap_applied": rho_cap_applied}
        return g_path, moat_label, rho, n_years

    def compute_terminal_growth(self, reinv, wacc):
        fcff_cagr = 0.0; h = self._fcff_history
        if h is not None and len(h) >= 4:
            h20 = h.dropna().tail(20); f0, fl, n = float(h20.iloc[0]), float(h20.iloc[-1]), len(h20)
            if f0 > 0 and fl > 0 and n >= 4: fcff_cagr = float(np.clip((fl / f0) ** (4.0 / n) - 1.0, -0.10, 0.15))
        roic = self.result_df["roic"].dropna(); roic_med = float(roic.median()) if not roic.empty else 0.06
        g_cap = min(self.gdp_growth, self.rf) if V8_TERMINAL_RF_CAP else self.gdp_growth
        g = float(np.clip(0.5 * fcff_cagr + 0.5 * roic_med * reinv, 0.0, g_cap))
        if self.verbose: log(self.ticker_dart, f"g_terminal={g:.4f} FCFF_CAGR={fcff_cagr:.4f} g_roic={roic_med*reinv:.4f}")
        return g

    # ── 6. Valuation ─────────────────────────────────────────
    def compute_valuation(self):
        re, bb = self.compute_beta_re(); tax = self.estimate_tax_rate(); wacc, re_used, rd = self.compute_wacc(re, tax)
        df = self.result_df.copy()
        pos = df[df["nopat"] > 0]["reinvestment_rate"].dropna(); reinv_med = float(self._winsorize(pos).median()) if not pos.empty else 0.3
        fq = df["fcff"].values
        ph1 = np.array([float(np.sum(fq[y:y+4])) for y in range(0, len(fq), 4)])
        ph2_g, moat, rho, n_ph2 = self._estimate_phase2_growth(wacc)
        ph2, last = [], float(ph1[-1])
        for g in ph2_g:
            last *= (1 + g); ph2.append(last)
        ph2 = np.array(ph2)
        g_term = self.compute_terminal_growth(reinv_med, wacc)
        if wacc - g_term < V8_MIN_TV_SPREAD: g_term = max(0.0, wacc - V8_MIN_TV_SPREAD)
        fl = float(ph2[-1]); tv = fl * (1 + g_term) / (wacc - g_term)
        tv_capped, tv_mult = False, np.nan
        if fl > 0:
            tv_mult = tv / fl
            if V8_TV_FCFF_MULT_CAP and tv > V8_TV_FCFF_MULT_CAP * fl:
                tv, tv_mult, tv_capped = V8_TV_FCFF_MULT_CAP * fl, float(V8_TV_FCFF_MULT_CAP), True
                if self.verbose: log(self.ticker_dart, f"[v8] TV 배수 상한 {V8_TV_FCFF_MULT_CAP:.0f}× 적용")
        all_annual = np.concatenate([ph1, ph2]); T = len(all_annual)
        pv_fcff = float(sum(all_annual[t] / (1 + wacc) ** (t + 1) for t in range(T))); pv_tv = tv / (1 + wacc) ** T
        ev = pv_fcff + pv_tv
        wide = self._fs_wide; cash = pd.Series(0.0, index=wide.index)
        for k in CASH_KEYS:
            if k in wide.columns: cash = cash + wide[k].fillna(0)
        nd = (self._total_debt_series() - cash).dropna(); net_debt = float(nd.iloc[-1]) if not nd.empty else 0.0
        equity_val = ev - net_debt
        mc_actual = self._mkt_cap if self._mkt_cap else np.nan
        mc_ratio = equity_val / mc_actual if (mc_actual and np.isfinite(mc_actual) and mc_actual > 0) else np.nan
        if V8_SANITY_GUARD and np.isfinite(mc_ratio) and (mc_ratio > V8_SANITY_MC_RATIO or (0 < mc_ratio < 1.0 / V8_SANITY_MC_RATIO)):
            raise ValueError(f"[{self.ticker_dart}] v8 sanity guard: 지분가치 {equity_val/1e12:,.2f}조 vs 시총 {mc_actual/1e12:,.2f}조 = {mc_ratio:,.1f}배 → 평가 제외")
        shares = self._estimate_shares()
        target_price = equity_val / shares if (not np.isnan(shares) and shares > 0) else np.nan
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE); self._current_price = cp
        upside = ((target_price / cp) - 1) * 100 if (cp and not np.isnan(target_price) and cp > 0) else np.nan
        self.valuation = {
            "ticker": self.ticker_dart, "wacc": wacc, "wacc_re": re_used, "wacc_rd": rd,
            "beta_raw": self._beta_info.get("beta_raw") if self._beta_info else np.nan, "beta_blume": bb,
            "g_terminal": g_term, "reinvestment_rate": reinv_med,
            "pv_fcff": pv_fcff, "terminal_value": tv, "pv_tv": pv_tv, "tv_weight_pct": pv_tv / ev * 100 if ev else np.nan,
            "enterprise_value": ev, "net_debt": net_debt, "equity_value": equity_val, "shares": shares, "shares_method": self._shares_method,
            "target_price": target_price, "current_price": cp, "upside_pct": upside,
            "ph1_annual": ph1.tolist(), "ph2_annual": ph2.tolist(), "ph2_growth": ph2_g, "all_annual": all_annual.tolist(), "t_total": T,
            "fcff_last_annual": fl, "moat_label": moat, "eva_spread": self._eva_cache.get("eva_spread", np.nan),
            "roic": self._eva_cache.get("roic", np.nan), "moat_rho": rho, "rho_cap_applied": self._eva_cache.get("rho_cap_applied", False),
            "tv_fcff_mult": tv_mult, "tv_capped": tv_capped, "mc_ratio": mc_ratio, "n_phase2": n_ph2,
            "eva_series": self._eva_cache.get("eva_series", []), "g0": self._eva_cache.get("g0", np.nan), "g0_method": self._eva_cache.get("g0_method", "?"),
            "g0_hist": self._eva_cache.get("g0_hist"), "g0_fc": self._eva_cache.get("g0_fc"),
            "nwc_method": self._nwc_method, "nwc_to_sales": self._nwc_to_sales, "op_current_assets": self._op_ca_latest, "op_current_liab": self._op_cl_latest,
            "revenue_quarters": len(self._sales_actual), "forecast_model": self._used_model, "forecast_date": self._forecast_date,
            "data_source": TABLE_DART_FS,
        }
        if self.verbose:
            tp = f"{target_price:,.0f}원" if not np.isnan(target_price) else "N/A"; cps = f"{cp:,.0f}원" if cp else "N/A"
            up = f"{upside:+.1f}%" if not np.isnan(upside) else "N/A"
            log(self.ticker_dart, f"3-Stage EV={ev/1e12:.2f}조 TP={tp} CP={cps} Up={up} TV비중={self.valuation['tv_weight_pct']:.1f}% Moat={moat}")
        return self

    def run(self):
        self.load_sales(); self.load_financials(); self.compute_fcff(); self.compute_valuation(); return self


print("[OK] DartDCFModel 정의 완료")

## PART 4 · Excel 출력 (DB 저장 없음 — 요구사항 6·7)

파일명: `{ticker}_{기업명}_FCFF_DART_{YYYYMMDD}.xlsx` → `EXPORT_DIR`

| 시트 | 내용 |
|---|---|
| Summary | 적정주가·EV·WACC·g·Moat·추정 방식(주식수 등)·데이터 출처 |
| FCFF_Forecast | 분기별 Sales→EBIT→NOPAT→FCFF (Phase 1) |
| Annual_Path | Phase1/Phase2 연간 FCFF, g 경로, TV |
| Sales | 실적 + 예측 매출 (모델별) |
| Forecast_Vintages | 계측일별 Ensemble 예측 변화 (요구사항 4 추적) |
| Financials_Q | DART 분기 wide (억원) |
| Account_Mapping | 표준키 ↔ DART 계정 채택 내역 · 누적/분기 모드 |
| Quality | DataQualityReport (fallback 추적) |
| Assumptions | 모델 파라미터 |

In [ ]:
def _corp_name(ticker: str) -> str:
    """파일명용 기업명: CORP_NAMES → 시총 테이블(name/corp_name 컬럼) → 'NA'."""
    if ticker in CORP_NAMES:
        return CORP_NAMES[ticker]
    for col_n in ("name", "corp_name", "company_name"):
        try:
            with engine.connect() as c:
                r = c.execute(_sa_text(f"SELECT {col_n} FROM {TABLE_MARKETCAP} WHERE ticker=:t ORDER BY date DESC LIMIT 1"),
                              {"t": to_dg_ticker(ticker)}).fetchone()
            if r and r[0]:
                return re.sub(r"[\\/:*?\"<>|\s]+", "_", str(r[0]))
        except Exception:
            continue
    return "NA"


def _quality_df(report) -> pd.DataFrame:
    for attr in ("to_dataframe", "to_df", "as_dataframe"):
        if hasattr(report, attr):
            try: return getattr(report, attr)()
            except Exception: pass
    for attr in ("entries", "items", "records", "rows"):
        v = getattr(report, attr, None)
        if v:
            try: return pd.DataFrame(v)
            except Exception: pass
    return pd.DataFrame({"report": [str(report)]})


def export_dart_excel(model: DartDCFModel, out_dir: Path = EXPORT_DIR, run_date: Optional[date] = None) -> Path:
    run_date = run_date or date.today(); v = model.valuation; tk = model.ticker_dart
    name = _corp_name(tk)
    path = out_dir / f"{tk}_{name}_{EXPORT_FILE_TAG}_{run_date:%Y%m%d}.xlsx"
    if path.exists(): path.unlink()

    def _pct(x): return None if x is None or (isinstance(x, float) and np.isnan(x)) else x
    summary = pd.DataFrame([
        ("종목코드", tk), ("기업명", name), ("데이터 출처", v["data_source"]), ("평가일", str(run_date)),
        ("매출예측 계측일 (forecast_date)", v["forecast_date"]), ("매출예측 모델", v["forecast_model"]),
        ("사용 실적 분기 수", v["revenue_quarters"]),
        ("──── 결과 ────", ""),
        ("현재가 (원)", v["current_price"]), ("적정주가 (원)", _pct(v["target_price"])), ("Upside (%)", _pct(v["upside_pct"])),
        ("EV (원)", v["enterprise_value"]), ("PV(FCFF) (원)", v["pv_fcff"]), ("PV(TV) (원)", v["pv_tv"]), ("TV 비중 (%)", v["tv_weight_pct"]),
        ("Net Debt (원)", v["net_debt"]), ("지분가치 (원)", v["equity_value"]),
        ("주식수 (추정)", _pct(v["shares"])), ("주식수 추정방식", v["shares_method"]),
        ("──── 할인율 ────", ""),
        ("WACC", v["wacc"]), ("Re", v["wacc_re"]), ("Rd", v["wacc_rd"]), ("β_raw", _pct(v["beta_raw"])), ("β_blume", v["beta_blume"]),
        ("Rf", RF), ("ERP", ERP), ("실효세율", model._coefs["tax"]),
        ("──── 성장 ────", ""),
        ("g_terminal", v["g_terminal"]), ("g0", v["g0"]), ("g0 방식", v["g0_method"]),
        ("Moat", v["moat_label"]), ("EVA spread", _pct(v["eva_spread"])), ("ROIC", _pct(v["roic"])),
        ("ρ", v["moat_rho"]), ("ρ 상한 적용", v["rho_cap_applied"]), ("Phase2 연수", v["n_phase2"]),
        ("TV/FCFF 배수", _pct(v["tv_fcff_mult"])), ("TV 배수 상한 적용", v["tv_capped"]), ("지분가치/시총", _pct(v["mc_ratio"])),
        ("──── 계수 ────", ""),
        ("D&A / Sales (α)", model._coefs["alpha_da"]), ("CapEx / Sales (β)", model._coefs["beta_capex"]),
        ("NWC / Sales (γ)", model._coefs["gamma_nwc"]), ("NWC 정의", v["nwc_method"]),
        ("재투자율(median)", v["reinvestment_rate"]),
    ], columns=["항목", "값"])

    fc_df = model.result_df.copy(); fc_df["date"] = pd.to_datetime(fc_df["date"]).dt.date
    ann = pd.DataFrame({"year_idx": range(1, v["t_total"] + 1), "fcff_annual": v["all_annual"],
                        "phase": ["Phase1"] * len(v["ph1_annual"]) + ["Phase2"] * len(v["ph2_annual"]),
                        "g_path": [np.nan] * len(v["ph1_annual"]) + list(v["ph2_growth"])})
    ann["discount_factor"] = 1 / (1 + v["wacc"]) ** ann["year_idx"]; ann["pv"] = ann["fcff_annual"] * ann["discount_factor"]
    ann = pd.concat([ann, pd.DataFrame([{"year_idx": "TV", "fcff_annual": v["terminal_value"], "phase": "Terminal",
                                         "g_path": v["g_terminal"], "discount_factor": ann["discount_factor"].iloc[-1], "pv": v["pv_tv"]}])], ignore_index=True)

    sales = pd.concat([model._sales_actual.rename("actual"), model._sales_forecast.rename("forecast")], axis=1)
    sales.index = pd.to_datetime(sales.index).date; sales.index.name = "quarter_end"
    try:
        fc_all = pd.read_sql(_sa_text(f"SELECT date, indicator, value FROM {TABLE_DART_FC} WHERE ticker=:t AND forecast_date=:d"),
                             engine, params={"t": tk, "d": v["forecast_date"]})
        if not fc_all.empty:
            pv = fc_all.pivot(index="date", columns="indicator", values="value"); pv.index = pd.to_datetime(pv.index).date
            sales = sales.join(pv, how="outer")
    except Exception:
        pass
    vint = load_forecast_vintages(tk, model.db_info)
    fin = (model._fs_wide / 1e8).round(2); fin.index = pd.to_datetime(fin.index).date; fin.index.name = "quarter_end (억원)"
    assumptions = pd.DataFrame([(k, str(globals()[k])) for k in [
        "FORECAST_QUARTERS", "MIN_DATA_PERIODS", "ENSEMBLE_AGG", "IS_FLOW_MODE", "CF_FLOW_MODE", "MIN_HISTORY", "WINSORIZE_LIMITS",
        "GDP_GROWTH", "G_FLOOR_MIN", "G_CEIL", "OLS_MIN_R2", "OLS_MIN_SAMPLES", "ERP_METHOD", "DAMODARAN_ERP_KR", "RD_DEFAULT",
        "RD_ANNUALIZE_QUARTERLY", "WACC_FLOOR", "WACC_CAP", "V8_RD_FLOOR_RF", "V8_WACC_FLOOR_ABS", "V8_MIN_TV_SPREAD", "V8_TERMINAL_RF_CAP",
        "V8_TV_FCFF_MULT_CAP", "V8_SANITY_GUARD", "V8_SANITY_MC_RATIO", "V8_RHO_ADAPTIVE_CAP", "DEBT_KEYS", "CASH_KEYS",
        "USE_OPERATING_NWC", "FS_UNIT_MULTIPLIER", "FORECAST_MODULE", "TABLE_DART_FS", "TABLE_DART_FC"]], columns=["param", "value"])

    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        summary.to_excel(xw, sheet_name="Summary", index=False)
        fc_df.to_excel(xw, sheet_name="FCFF_Forecast", index=False)
        ann.to_excel(xw, sheet_name="Annual_Path", index=False)
        sales.to_excel(xw, sheet_name="Sales")
        (vint if not vint.empty else pd.DataFrame({"info": ["vintage 없음"]})).to_excel(xw, sheet_name="Forecast_Vintages")
        fin.to_excel(xw, sheet_name="Financials_Q")
        (model._mapping_df if model._mapping_df is not None else pd.DataFrame()).to_excel(xw, sheet_name="Account_Mapping", index=False)
        _quality_df(model.report).to_excel(xw, sheet_name="Quality", index=False)
        assumptions.to_excel(xw, sheet_name="Assumptions", index=False)
        for ws in xw.book.worksheets:
            ws.freeze_panes = "B2"
            for col in ws.columns:
                w = max((len(str(c.value)) for c in col if c.value is not None), default=8)
                ws.column_dimensions[col[0].column_letter].width = min(max(10, w + 2), 60)
    return path

## 실행
### Step A · (선택) 개별 종목 점검 — DART 계정 매핑 + 예측 미리보기 (DB 저장 X)

In [ ]:
INSPECT_TICKER = TICKERS[0]
_w, _mp = show_dart_account_coverage(INSPECT_TICKER, db_info)
_ins = inspect_dart_ticker(INSPECT_TICKER, db_info)

### Step B · 매출 예측 → `korea_revenue_forecast_from_DART` 저장 (forecast_date = 계측일)

In [ ]:
fc_result = run_dart_forecast_batch(TICKERS, db_info, forecast_date=FORECAST_DATE, rerun=RERUN_FORECAST, verbose=VERBOSE)

### Step C · FCFF Valuation → Excel (DB 저장 없음)

In [ ]:
run_date = date.today()
_summary_rows, t0 = [], time.time()
print(f"\n{'='*70}\n[DART FCFF Valuation] {len(TICKERS)}종목  run_date={run_date}  → {EXPORT_DIR}\n{'='*70}")
for i, tk in enumerate(TICKERS, 1):
    print(f"\n[{i}/{len(TICKERS)}] {tk} " + "─" * 50)
    try:
        _m = DartDCFModel(ticker=tk, engine=engine, db_info=db_info, rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX,
                          forecast_date=None, verbose=VERBOSE).run()
        _p = export_dart_excel(_m, EXPORT_DIR, run_date)
        v = _m.valuation
        tp = f"{v['target_price']:,.0f}원" if not np.isnan(v["target_price"]) else "N/A"
        cp = f"{v['current_price']:,.0f}원" if v["current_price"] else "N/A"
        up = f"{v['upside_pct']:+.1f}%" if not np.isnan(v["upside_pct"]) else "N/A"
        print(f"  ✓ Saved: {_p}")
        print(f"    적정주가 {tp}   현재가 {cp}   Upside {up}   WACC {v['wacc']:.2%}   g_term {v['g_terminal']:.2%}")
        print(f"    Moat {v['moat_label']} (ρ={v['moat_rho']:.3f}, Phase2={v['n_phase2']}년)   주식수: {v['shares_method']}   forecast_date={v['forecast_date']}")
        _summary_rows.append({"ticker": tk, "status": "ok", "excel": str(_p), "target_price": v["target_price"], "current_price": v["current_price"],
                              "upside_pct": v["upside_pct"], "wacc": v["wacc"], "g_terminal": v["g_terminal"], "moat": v["moat_label"],
                              "forecast_date": v["forecast_date"]})
    except Exception as e:
        print(f"  ✗ FAIL: {e}"); traceback.print_exc()
        _summary_rows.append({"ticker": tk, "status": "fail", "excel": "", "target_price": np.nan, "current_price": np.nan,
                              "upside_pct": np.nan, "wacc": np.nan, "g_terminal": np.nan, "moat": "?", "forecast_date": None, "msg": str(e)[:150]})
    finally:
        clear_memory()
print(f"\n{'='*70}\n[완료] OK={sum(r['status']=='ok' for r in _summary_rows)} FAIL={sum(r['status']=='fail' for r in _summary_rows)}  경과 {time.time()-t0:.0f}s\n{'='*70}")
summary_df = pd.DataFrame(_summary_rows).set_index("ticker")
display(summary_df)

### Step D · 계측일별 매출 예측 변화 추적 (요구사항 4)

행 = `forecast_date`(계측일), 열 = 예측 대상 분기. 계측일이 쌓일수록 같은 분기에 대한 예측이 어떻게 수정됐는지 한눈에 비교.

In [ ]:
VINTAGE_TICKER = TICKERS[0]
_v = load_forecast_vintages(VINTAGE_TICKER, db_info, indicator="Ensemble")
if _v.empty:
    print(f"[{VINTAGE_TICKER}] 저장된 vintage 없음")
else:
    print(f"[{VINTAGE_TICKER}] Ensemble 예측 (억원) — 계측일 × 대상분기")
    _disp = _v.copy(); num = _disp.columns.drop("last_actual")
    _disp[num] = (_disp[num] / 1e8).round(0)
    display(_disp)
    if len(_v) >= 2:
        _chg = ((_v[num].iloc[-1] / _v[num].iloc[-2] - 1) * 100).dropna().round(1)
        print(f"\n최근 계측({_v.index[-1]}) vs 직전({_v.index[-2]}) 변화율(%):"); print(_chg.to_string())